# HydroPulse model training, version 2 (Colab, free GPU)
## What your first run showed
Your first model was **not** a failure. Against the physics-only engine it was clearly better at surprise failures (97 / 94 / 87% fewer than a tuned counter, against 80 / 79 / 68%), at warning early enough (98.8 / 91.2 / 98.8% against 88.8 / 70.0 / 93.8%) and at finding a spent carbon filter (59 / 50 / 48% against 30 / 20 / 25%). It was **worse** at filter life used (66.8 / 69.3 / 60.9% against 75.9 / 77.2 / 67.4%) and at false alarms. That means it simply sat at a more cautious point on the same trade-off, so the single-row comparison could not say which model is truly better.

## What this version changes
1. **A fairer test.** It compares failures **at the same filter-life use** (a matched operating point), which removes the trade-off from the comparison.
2. **A better model.** The first model looked at one snapshot of 15 numbers. This one is a **streaming GRU** (a small recurrent network) that reads the whole history day by day. It can notice when a wear speed-up started, how large it was, and how long a water event lasted, which a snapshot throws away. It still takes the physics numbers as inputs (a physics-informed hybrid), so it corrects the physics instead of replacing it.
3. **Proper outputs.** The range outputs cannot cross (the 10% value is always below the 50% value), and the ranges are calibrated separately for short, medium and long horizons.
4. **A model shoot-out.** It also trains a gradient-boosted tree model (LightGBM) and your first network on the same data, so the choice of model is measured, not assumed.

## Which hardware
Use the free **T4 GPU** (Runtime > Change runtime type > T4 GPU). A **TPU is not a good fit**: this model is small and processes days in sequence, which TPUs handle poorly, and it would need extra setup for no speed-up.

## How long it takes (estimates, free T4)
| Step | Time |
|---|---|
| Setup and data download | 2 min |
| Generate training data (CPU, 30 seeds x 3 worlds; I timed 1 seed at about 3 s) | 2 to 4 min |
| Train the GRU (GPU, up to 40 epochs, stops early if it stops improving) | 10 to 18 min |
| Model shoot-out (LightGBM on CPU) | 3 to 6 min |
| Benchmark both engines (physics about 1 min, GRU about 6 to 12 min) | 7 to 13 min |
| **Total** | **about 25 to 45 min** |

Set `QUICK = True` in the data cell for a rehearsal of about 12 minutes (fewer seeds and epochs, lower quality). Only the data-generation time was measured; the training, shoot-out and benchmark times are my estimates.

**Before you start:** put `HydroPulse_Final_Prototype.zip` (the new one) into My Drive > atomquest. All results are saved back to that folder.

In [ ]:
# 1. Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, subprocess, urllib.request, zipfile, glob, time
BASE = '/content/drive/MyDrive/atomquest'
for d in ('data', 'models', 'code'): os.makedirs(f'{BASE}/{d}', exist_ok=True)
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (switch to a T4 GPU runtime)')

In [ ]:
# 2. Unpack the NEW project zip (it contains the GRU-ready engine)
z = glob.glob(f'{BASE}/**/HydroPulse_Final_Prototype.zip', recursive=True)
assert z, 'Upload the new HydroPulse_Final_Prototype.zip into My Drive/atomquest/ first.'
import shutil; shutil.rmtree(f'{BASE}/code/hp', ignore_errors=True)
zipfile.ZipFile(z[0]).extractall(f'{BASE}/code')
HP = f'{BASE}/code/hp'; sys.path.insert(0, HP); os.chdir(HP)
assert 'GruRunner' in open(f'{HP}/engine.py').read(), 'This is the old zip. Upload the new one.'
print(sorted(os.listdir(HP)))

## Dataset (real, public, free)
Real 540-day record of an operating reverse-osmosis plant (daily RO product volume). Page: https://github.com/Asaeel/ro-fouling-early-warning
Direct file: https://raw.githubusercontent.com/Asaeel/ro-fouling-early-warning/main/data/MBR_RO_AIOT_daily_flow_540d.xlsx
It supplies the **real noise and outage pattern** used to drive the simulator. It has flow only, so it cannot teach lifetime prediction on its own.

In [ ]:
# 3. Download the real dataset to Drive and refresh the real-noise file
import statistics as st
URL = 'https://raw.githubusercontent.com/Asaeel/ro-fouling-early-warning/main/data/MBR_RO_AIOT_daily_flow_540d.xlsx'
dst = f'{BASE}/data/MBR_RO_AIOT_daily_flow_540d.xlsx'
if not os.path.exists(dst): urllib.request.urlretrieve(URL, dst)
%pip install -q openpyxl
import openpyxl
ws = openpyxl.load_workbook(dst, read_only=True, data_only=True).worksheets[0]
q = [r[4] for r in ws.iter_rows(min_row=3, values_only=True) if isinstance(r[0], int)]; res = []
for i in range(7, len(q) - 7):
    m = st.median(q[i-7:i+8])
    if m > 0 and q[i] > .3 * m: res.append(round(q[i] / m - 1, 4))
json.dump(dict(rel_resid=res), open(f'{HP}/data/real_noise.json', 'w')); print(len(q), 'days;', len(res), 'real noise samples')

In [ ]:
# 4. Generate full day-by-day training sequences from the simulator (physics-only engine, no model)
QUICK = False                      # True = 15-minute rehearsal
TRAIN_SEEDS = range(1000, 1008) if QUICK else range(1000, 1030)
VAL_SEEDS = range(1050, 1052) if QUICK else range(1050, 1056)     # the benchmark uses seed 7: never seen here
import numpy as np, engine, evaluate as ev
engine.MODEL = None; DAYS = ev.DAYS
def gen_seq(seeds):
    Xs, Rs, Cs = [], [], []
    for sd in seeds:
        for law, real in (('linear', False), ('harder', False), ('linear', True)):
            H = ev.make_env(sd, real); S, info = ev.simulate(H, law, 'none', 0, True)
            rem = np.full((len(H), DAYS), -1., 'float32')
            for i, inf in enumerate(info):
                T = inf['fail']
                if T is not None: rem[i, :T] = T - np.arange(T)
            Xs.append(S['cap']); Rs.append(rem); Cs.append(S['car'])
    return np.concatenate(Xs), np.concatenate(Rs), np.concatenate(Cs)
t0 = time.time(); Xtr, Rtr, Ctr = gen_seq(TRAIN_SEEDS); print('train sequences', Xtr.shape, round(time.time() - t0), 's')
Xva, Rva, Cva = gen_seq(VAL_SEEDS); print('validation sequences', Xva.shape)

In [ ]:
# 5. Train the streaming GRU on the GPU
import torch, torch.nn as nn, torch.nn.functional as Fn
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
mu = Xtr.reshape(-1, 19).mean(0); sd = Xtr.reshape(-1, 19).std(0) + 1e-6
def prep(X, R, C):
    return [torch.clamp(torch.tensor((X - mu) / sd), -8, 8), torch.tensor(np.log1p(np.maximum(R, 0))),
            torch.tensor(((R >= 0) & (X[:, :, 14] * 100 < 150)).astype('float32')), torch.tensor(C),
            torch.tensor((np.arange(DAYS)[None, :] >= 60).astype('float32')).expand(len(X), -1).contiguous()]
Ttr, Tva = prep(Xtr, Rtr, Ctr), prep(Xva, Rva, Cva)
class Net(nn.Module):
    def __init__(s, nin=19, H=64):
        super().__init__(); s.g = nn.GRU(nin, H, batch_first=True); s.h1 = nn.Linear(H, 32); s.h2 = nn.Linear(32, 4)
    def forward(s, x): o, _ = s.g(x); return s.h2(torch.relu(s.h1(o)))
def parts(o):                                    # non-crossing quantiles: q10 <= q50 <= q90
    a = o[..., 0]; return a - Fn.softplus(o[..., 1]), a, a + Fn.softplus(o[..., 2]), o[..., 3]
def loss_fn(o, y, rm, c, cm):
    q = parts(o); L = 0
    for p, qq in zip(q[:3], (.1, .5, .9)): e = y - p; L = L + torch.maximum(qq * e, (qq - 1) * e)
    lr = (L / 3 * rm).sum() / rm.sum().clamp(min=1)
    lc = (Fn.binary_cross_entropy_with_logits(q[3], c, reduction='none') * cm).sum() / cm.sum()
    return lr + lc
net = Net().to(dev); EPOCHS = 12 if QUICK else 40; BS = 96
opt = torch.optim.AdamW(net.parameters(), 3e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, 3e-3, total_steps=EPOCHS * ((len(Ttr[0]) + BS - 1) // BS))
best, bad, state, t0 = 1e9, 0, None, time.time()
for ep in range(EPOCHS):
    net.train(); perm = torch.randperm(len(Ttr[0]))
    for i in range(0, len(perm), BS):
        b = perm[i:i + BS]; xb = [t[b].to(dev) for t in Ttr]
        opt.zero_grad(); loss_fn(net(xb[0]), *xb[1:]).backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sched.step()
    net.eval(); vl = 0.
    with torch.no_grad():
        for i in range(0, len(Tva[0]), 256):
            xb = [t[i:i + 256].to(dev) for t in Tva]; vl += loss_fn(net(xb[0]), *xb[1:]).item() * len(xb[0])
    vl /= len(Tva[0]); print('epoch', ep, 'validation loss', round(vl, 4), round(time.time() - t0), 's')
    if vl < best: best, bad, state = vl, 0, {k: v.clone() for k, v in net.state_dict().items()}
    else:
        bad += 1
        if bad >= 8: break
net.load_state_dict(state); print('best validation loss', round(best, 4))

In [ ]:
# 6. Calibrate the ranges (separately for short, medium, long horizons), export, and PROVE the export is exact
def predict(T):
    outs = []
    with torch.no_grad():
        for i in range(0, len(T[0]), 256): outs.append(torch.stack(parts(net(T[0][i:i + 256].to(dev))), -1).cpu())
    return torch.cat(outs).numpy()                          # [sequences, days, (q10, q50, q90, carbon logit)]
P = predict(Tva); rp = Xva[:, :, 14] * 100; y = np.log1p(np.maximum(Rva, 0)); mk = (Rva >= 0) & (rp < 150); off = []
for a, b in ((0, 30), (30, 90), (90, 150)):
    m = mk & (rp >= a) & (rp < b); off.append([float(np.quantile((y - P[..., 0])[m], .10)), float(np.quantile((y - P[..., 2])[m], .90))])
lo, hi = P[..., 0] + np.array(off)[:, 0][np.digitize(rp, [30, 90])], P[..., 2] + np.array(off)[:, 1][np.digitize(rp, [30, 90])]
print('range holds the truth', round(100 * np.mean(((y >= lo) & (y <= hi))[mk])), '% (target 80%)')
print('truth never earlier than the safe date', round(100 * np.mean((y >= lo)[mk])), '% (target 90%)')
from sklearn.metrics import roc_auc_score
cm = np.broadcast_to(np.arange(DAYS)[None, :] >= 60, Cva.shape); sub = np.random.RandomState(0).rand(*Cva.shape) < .1
print('carbon-spent AUC', round(roc_auc_score(Cva[cm & sub], P[..., 3][cm & sub]), 3))
g = net.g; sd_ = {k: v.detach().cpu().numpy().tolist() for k, v in net.state_dict().items()}
model = dict(type='gru', mean=mu.tolist(), std=sd.tolist(), Wih=sd_['g.weight_ih_l0'], Whh=sd_['g.weight_hh_l0'], bih=sd_['g.bias_ih_l0'], bhh=sd_['g.bias_hh_l0'],
             head=[dict(W=sd_['h1.weight'], b=sd_['h1.bias']), dict(W=sd_['h2.weight'], b=sd_['h2.bias'])], off=off)
# exactness check: the app's runner must reproduce the trained network step by step
R = engine.GruRunner(model); h = R.init(); xs = Xva[0, :300]; mx = 0.
with torch.no_grad(): ref = net(torch.clamp(torch.tensor((xs - mu) / sd)[None], -8, 8).to(dev))[0].cpu().numpy()
for t in range(300): h = R.step(h, xs[t].tolist()); mx = max(mx, max(abs(a - b) for a, b in zip(R.out(h), ref[t])))
print('max difference between trained network and app runner:', mx); assert mx < 1e-3, 'Export mismatch: do not use this model.json'
json.dump(model, open(f'{BASE}/models/model_gru.json', 'w')); json.dump(model, open(f'{HP}/model.json', 'w'))
print('saved model_gru.json', round(os.path.getsize(f'{HP}/model.json') / 1024), 'KB')

In [ ]:
# 7. Model shoot-out on held-out validation sequences (same rows for every model)
%pip install -q lightgbm
import lightgbm as lgb
def snap(X, R, C, step=7):
    d = np.arange(60, DAYS, step); x = X[:, d].reshape(-1, 19); r = R[:, d].reshape(-1); c = C[:, d].reshape(-1); m = (r >= 0) & (x[:, 14] * 100 < 150)
    return x, r, c, m
xt, rt, ct, mt = snap(Xtr, Rtr, Ctr); xv, rv, cv, mv = snap(Xva, Rva, Cva); yt, yv = np.log1p(np.maximum(rt, 0)), np.log1p(np.maximum(rv, 0))
pin = lambda y, p, q: float(np.mean(np.maximum(q * (y - p), (q - 1) * (y - p))))
rows = {}
rows['Physics only'] = (float(np.median(np.abs(np.log1p(xv[mv, 14] * 100) - yv[mv]))), None, float(roc_auc_score(cv, xv[:, 5])))
t0 = time.time(); Q = {}
for a in (.1, .5, .9):
    Q[a] = lgb.LGBMRegressor(objective='quantile', alpha=a, n_estimators=300, learning_rate=.08, num_leaves=63, subsample=.8, subsample_freq=1, verbose=-1).fit(xt[mt][:300000], yt[mt][:300000])
clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=.08, num_leaves=63, verbose=-1).fit(xt[::2], ct[::2])
pq = {a: Q[a].predict(xv[mv]) for a in Q}
rows['LightGBM trees'] = (float(np.median(np.abs(pq[.5] - yv[mv]))), np.mean([pin(yv[mv], pq[a], a) for a in Q]), float(roc_auc_score(cv, clf.predict_proba(xv)[:, 1])))
print('trees trained in', round(time.time() - t0), 's')
d_idx = np.arange(60, DAYS, 7); Pg = P[:, d_idx].reshape(-1, 4)
rows['GRU (this notebook)'] = (float(np.median(np.abs(Pg[mv, 1] - yv[mv]))), np.mean([pin(yv[mv], Pg[mv, j], a) for j, a in enumerate((.1, .5, .9))]), float(roc_auc_score(cv, Pg[:, 3])))
print(f"{'model':24s}{'median log error':>20s}{'pinball loss':>16s}{'carbon AUC':>14s}   (lower, lower, higher is better)")
for k, (a, b, c) in rows.items(): print(f"{k:24s}{a:20.3f}{('-' if b is None else round(b, 3)):>16}{c:14.3f}")

In [ ]:
# 8. The real test: full benchmark with and without the model, then compare at the SAME filter-life use
def run(args, out):
    r = subprocess.run([sys.executable, 'evaluate.py'] + args + ['--out', out], capture_output=True, text=True, cwd=HP)
    if r.returncode: print(r.stderr[-1500:])
    return json.load(open(f'{HP}/{out}'))
t0 = time.time(); phys = run(['--no-model'], 'results_physics.json'); print('physics benchmark', round(time.time() - t0), 's')
t0 = time.time(); gru = run([], 'results_gru.json'); print('GRU benchmark', round(time.time() - t0), 's')
print(f"{'measure':58s}{'physics':>22s}{'GRU':>22s}   (normal / harder / real-noise)")
for a, b in zip(phys['rows'], gru['rows']): print(f"{a['name'][:56]:58s}{str(a['vals']):>22s}{str(b['vals']):>22s}")
def fails_at_util(res, world, u):
    pts = sorted((v[world][1], v[world][0]) for v in res['curve'].values()); return float(np.interp(u, [p[0] for p in pts], [p[1] for p in pts]))
print('\nSame filter-life use, how many surprise failures? (fewer is better)')
wins = 0
for w in ('standard', 'harder', 'stress'):
    pf, pu = phys['detail'][w]['hydro']['fail'], phys['detail'][w]['hydro']['util']
    gf = fails_at_util(gru, w, pu); wins += gf <= .8 * pf
    print(f'{w:9s} physics: {pf} failures at {pu}% life used | GRU at the same {pu}%: {gf:.0f} failures')
cov = gru['rows'][4]['vals']; ok_cov = all(v >= t for v, t in zip(cov, (78, 76, 70)))
print('\nVERDICT:', 'GRU is better at matched life use in', wins, 'of 3 worlds; range reliability', 'OK' if ok_cov else 'WEAKER')

In [ ]:
# 9. Adopt only if it truly wins, then save everything to Drive
ADOPT = None                       # None = automatic rule (wins in at least 2 of 3 worlds and range reliability OK); True/False to override
decision = (wins >= 2 and ok_cov) if ADOPT is None else ADOPT
if decision:
    shutil.copy(f'{HP}/results_gru.json', f'{HP}/results.json'); shutil.copy(f'{HP}/results_gru.md', f'{HP}/results.md'); print('GRU ADOPTED: results.json now shows the GRU results.')
else:
    os.remove(f'{HP}/model.json'); print('GRU NOT adopted: model.json removed from the project; a copy stays in Drive/models/model_gru.json.')
for f in ('results_physics.json', 'results_gru.json'): shutil.copy(f'{HP}/{f}', f'{BASE}/models/{f}')
out = f'{BASE}/HydroPulse_Final_Prototype_trained.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(HP):
        for f in files:
            if '__pycache__' not in root: zf.write(os.path.join(root, f), os.path.join('hp', os.path.relpath(os.path.join(root, f), HP)))
print('Download from Drive:', out)